# Seaquest Stage-H0 — Hostile-Field Qualification (Colab, PyTorch only)

**Do NOT install/run ALE, ROMs, OCAtari, EnvPool, JAX, Flax, or the HF teacher here.** This notebook only loads the FROZEN `raw_hf` + hostile metadata pack (produced locally in Docker), verifies every raw SHA256, builds the hostile-removed four-frame states, runs the three probes and writes the qualification report. It is THIN: every cell calls a committed module under `seaquest_ccrl/hostile` / `seaquest_ccrl/scripts`.

In [ ]:
# 0. environment (torch only) + repo on path
#    The Stage-H0 code lives on the 'seaquest-stage-h0' BRANCH (not main).
import sys, os, subprocess
REPO = os.environ.get('H0_REPO', '/content/Goal-Conditioned-Confounded-MsPacman')
BRANCH = os.environ.get('H0_BRANCH', 'seaquest-stage-h0')
REPO_URL = os.environ.get('H0_REPO_URL',
    'https://github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman.git')
if not os.path.isdir(os.path.join(REPO, 'seaquest_stage_h0')):
    if not os.path.isdir(os.path.join(REPO, '.git')):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO], check=True)
    else:  # repo cloned on the wrong branch (e.g. main) -> switch
        subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=True)
sys.path.insert(0, REPO)
assert os.path.isdir(os.path.join(REPO, 'seaquest_stage_h0')), (
    f'seaquest_stage_h0 not found under {REPO!r}; clone branch {BRANCH!r}')
RAW = os.environ.get('H0_RAW', f'{REPO}/seaquest_ccrl/data/raw_hf')
META = os.environ.get('H0_META', f'{REPO}/seaquest_ccrl/data/hostile_h0_metadata')
OUT = os.environ.get('H0_OUT', f'{REPO}/artifacts/seaquest/hostile_h0')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo OK:', REPO, '| branch:', BRANCH)
# NOTE: raw_hf frames + hostile_h0_metadata are DATA (untracked, not in the repo).
# Upload them to Drive and point H0_RAW / H0_META at those paths.

In [ ]:
# 1. verify every raw SHA256 + pack self-consistency (no teacher/ALE)
from seaquest_stage_h0.validate_recollection_parity import validate
pv = validate(RAW, META, os.path.join(OUT, 'recollection_parity.json'))
assert pv['ok'], 'HOSTILE_RECOLLECTION pack invalid — see recollection_parity.json'
pv['totals']

In [ ]:
# 2. build hostile-removed four-frame states (verifies removal invariants per frame)
#    uses the RESOLVED palette (observed RGB) from the local object-identity audit
from seaquest_ccrl.hostile.data import HostileH0Data
from seaquest_ccrl.hostile import removal as RM
import os, json
pal_path = os.path.join(OUT, 'schema', 'resolved_palette.json')
palettes = RM.load_resolved_palettes(pal_path) if os.path.exists(pal_path) else None
tol = json.load(open(pal_path))['resolved_tol'] if os.path.exists(pal_path) else RM.DEFAULT_TOL
data = HostileH0Data(RAW, META, device=DEVICE, load_visible=True, verify_sha=True,
                     palettes=palettes, tol=tol)
print('N', data.N, 'episodes', data.n_ep, 'palette', 'resolved' if palettes else 'RAM-default')

In [ ]:
# 3. support / data density (Section 12)
import json
sup = data.support_summary()
json.dump(sup, open(os.path.join(OUT, 'support.json'), 'w'), indent=2)
sup['enemy']['pass'], sup['missile']['pass'], sup['joint']['pass']

In [ ]:
# 4. Probe A — hiddenness (P0 / PV / PM)
from seaquest_ccrl.scripts import h0_probe_hiddenness as PA
hidden = PA.run(data, os.path.join(OUT, 'hiddenness'), device=DEVICE)
{c: hidden[c]['recovery_ci'] for c in hidden}

In [ ]:
# 5. Probe B — conditional U -> action
from seaquest_ccrl.scripts import h0_probe_action as PB
action = PB.run(data, os.path.join(OUT, 'action'), device=DEVICE)
{c: (action[c]['improvement_mean'], action[c]['improvement_ci']) for c in action}

In [ ]:
# 6. Probe C — conditional U -> future
from seaquest_ccrl.scripts import h0_probe_future as PC
future = PC.run(data, os.path.join(OUT, 'future'), device=DEVICE)
{c: list(future[c]['per_horizon'].keys()) for c in future}

In [ ]:
# 7. assemble component results -> qualification report
from seaquest_ccrl.scripts import h0_qualify as Q
def hidden_in(c):
    h = hidden.get(c);
    return None if h is None else {'recovery_ci': h['recovery_ci'],
        'visible_better_than_prior': h['visible_better_than_prior'],
        'masked_nearest_r2': h['masked_nearest_r2'], 'adequate_support': h['adequate_support']}
results = {
  'recollection': {'identical': True},  # asserted at Docker collection time
  'object_schema': json.load(open(os.path.join(OUT,'schema','object_identity_audit.json'))),
  'removal': json.load(open(os.path.join(OUT,'removal','removal_audit.json'))),
  'support': {c: sup[c] for c in ('enemy','missile','joint')},
  'hiddenness': {c: hidden_in(c) for c in ('enemy','missile')},
  'action': {c: action.get(c) for c in ('enemy','missile','joint')},
  'future': {c: future.get(c) for c in ('enemy','missile','joint')},
}
# joint hiddenness defaults to enemy (joint removal == enemy+missile removed)
results['hiddenness']['joint'] = hidden_in('enemy')
rep = Q.decide(results)
Q.write_report(rep, os.path.join(OUT,'hostile_qualification.json'), os.path.join(OUT,'hostile_qualification.md'))
print('FINAL OUTCOME:', rep['final_outcome'], '(', rep['failure_kind'], ')')
rep['components']

## Pack + download all Stage-H0 results
Zips everything under `OUT` (qualification report, per-probe raw predictions/losses, support, and the copied audit JSONs) and triggers a browser download in Colab.

In [ ]:
# 8. pack + download all results
import shutil, zipfile, os
stamp = os.environ.get('H0_STAMP', 'results')
pack_path = shutil.make_archive(f'/content/seaquest_hostile_h0_{stamp}', 'zip', OUT)
size_mb = round(os.path.getsize(pack_path) / 1e6, 2)
with zipfile.ZipFile(pack_path) as z:
    names = z.namelist()
print(f'packed {pack_path}  ({size_mb} MB, {len(names)} files)')
for n in sorted(names):
    if n.endswith(('.json', '.md', '.png', '.csv')):
        print('  ', n)
try:
    from google.colab import files
    files.download(pack_path)
except Exception as e:
    print('(not in Colab / download unavailable):', e, '-> grab', pack_path)

## Smoke (engineering only — NOT a scientific result)
Runs the qualification wiring on MOCK metrics; writes to a `smoke/` dir, never the real report.

In [ ]:
from seaquest_stage_h0.build_hostile_h0_notebook import synthetic_smoke
synthetic_smoke(os.path.join(OUT, 'smoke'))